# XAU/EUR Gold Prediction — Interactive Notebook

Run each cell in order to execute the full pipeline interactively.

```
pip install -r ../requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import config as cfg
from data.fetcher import GoldDataFetcher
from data.features import FeatureEngineer
from sentiment.grok_client import ManualSentimentInjector

print('Setup complete.')

## Step 1: Fetch & Prepare Data

In [ ]:
# Fetch raw data (uses cache if available)
fetcher = GoldDataFetcher(cfg, cache_dir=cfg.DATA_DIR)
raw_df  = fetcher.fetch_all(use_cache=True)

print(f'Shape: {raw_df.shape}')
print(f'Date range: {raw_df.index[0].date()} → {raw_df.index[-1].date()}')
raw_df[['XAU_EUR', 'EUR_USD', 'VIX']].tail(10)

In [ ]:
# Plot XAU/EUR price history
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(raw_df.index, raw_df['XAU_EUR'], color='gold', lw=1.5)
axes[0].set_ylabel('XAU/EUR (€)')
axes[0].set_title('XAU/EUR Gold Price History', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

if 'VIX' in raw_df.columns:
    axes[1].fill_between(raw_df.index, raw_df['VIX'], color='red', alpha=0.4)
    axes[1].set_ylabel('VIX (Fear Index)')
    axes[1].grid(alpha=0.3)

if 'EUR_USD' in raw_df.columns:
    axes[2].plot(raw_df.index, raw_df['EUR_USD'], color='blue', lw=1)
    axes[2].set_ylabel('EUR/USD')
    axes[2].grid(alpha=0.3)
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Shade training vs test periods
for ax in axes:
    ax.axvspan(pd.Timestamp(cfg.TEST_START), pd.Timestamp(cfg.TEST_END),
               alpha=0.1, color='green', label='Test Period')

axes[0].legend(['XAU/EUR', 'Test Period (2023-2026)'], loc='upper left')
plt.tight_layout()
plt.savefig('../results/price_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build feature matrix
injector = ManualSentimentInjector()
sentiment_df = injector.to_dataframe(cfg.TRAIN_START, cfg.TEST_END)

engineer = FeatureEngineer(cfg)
feature_df = engineer.build(raw_df, sentiment_df=sentiment_df)

print(f'Features: {len(engineer.feature_names)}')
print('Feature list:', engineer.feature_names)
feature_df[engineer.feature_names].describe()

In [ ]:
# Feature correlation heatmap
fig, ax = plt.subplots(figsize=(16, 12))
corr = feature_df[engineer.feature_names].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdYlGn', center=0,
            vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../results/feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 2: Train Models (2012–2022)

In [ ]:
from training.trainer import Trainer

trainer = Trainer(cfg)
trainer.feature_engineer.feature_names = engineer.feature_names

# Train all horizons (this takes time — ~30-60 mins for all deep models)
# To test quickly, train only 'next_day':
ensembles = trainer.train(
    feature_df = feature_df,
    horizons   = ['next_day'],   # Add more horizons when ready
    save_dir   = cfg.MODEL_DIR,
)
print('Training complete.')

## Step 3: Evaluate (2023–2026)

In [ ]:
from evaluation.evaluator import ModelEvaluator

evaluator = ModelEvaluator(cfg)
summary = evaluator.evaluate(
    feature_df    = feature_df,
    ensembles     = ensembles,
    feature_names = engineer.feature_names,
    verbose       = True,
)
summary

In [ ]:
# Plot cumulative returns
results = evaluator.get_results()
test_df = feature_df[feature_df.index >= pd.Timestamp(cfg.TEST_START)]

if 'next_day' in results:
    r = results['next_day']
    signals = r['signals']
    fwd_ret = test_df.get('next_day_return', pd.Series(0, index=test_df.index)).values

    n = min(len(signals), len(fwd_ret))
    strat = np.where(signals[:n]=='BUY', fwd_ret[:n],
            np.where(signals[:n]=='SELL', -fwd_ret[:n], 0))
    strat -= cfg.BROKER_SPREAD_PCT

    cum_strat = np.cumprod(1 + strat)
    cum_bnh   = np.cumprod(1 + fwd_ret[:n])

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(test_df.index[:n], cum_strat, label='Strategy', color='green', lw=2)
    ax.plot(test_df.index[:n], cum_bnh,   label='Buy & Hold', color='gold', lw=2)
    ax.axhline(1.0, color='grey', lw=0.5, linestyle='--')
    ax.set_title('Cumulative Returns: Strategy vs Buy & Hold (Test Period)', fontsize=13)
    ax.set_ylabel('Cumulative Return')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../results/cumulative_returns.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 4: Predict Today

In [ ]:
from prediction.predictor import GoldPredictor

predictor = GoldPredictor(cfg)
result = predictor.predict_now(
    feature_df    = feature_df,
    ensembles     = ensembles,
    feature_names = engineer.feature_names,
)
predictor.print_prediction(result)

## Step 5: Fine-Tune with Latest Data

In [ ]:
# After live trading: feed back actual outcomes to update the model
# This uses the last 90 days of data to fine-tune

trainer.finetune(
    feature_df   = feature_df,
    ensembles    = ensembles,
    lookback_days= 90,
    retrain_dl   = False,  # Set True for full deep model update
)
print('Fine-tuning complete. Models updated and saved.')